# Python challenges 7: graphs and grids

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-challenges-07-graphs-and-grids.ipynb)

Depth-first and breadth-first search, union-find and topological ordering. Grids are graphs whose neighbours are the four cells next to a cell. Statements are paraphrased, and each solution is collapsed under its exercise.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: Number of Islands

**Difficulty:** Medium · about 20 min

`grid` is a list of rows of `"1"` (land) and `"0"` (water). Count the islands, where an island is a group of land cells joined up, down, left or right.

In a pipeline: this is connected-component labelling, the same grouping that entity resolution uses to cluster records that are linked to each other.

**Example 1**

```text
Input:  grid = ["11000", "11000", "00100", "00011"]
Output: 3
```

**Example 2**

```text
Input:  grid = ["111", "010", "111"]
Output: 1
```

<details>
<summary>💡 Tip</summary>

Scan every cell. When you find unvisited land, count one island and flood-fill everything connected to it, up, down, left and right, marking cells as visited. Recursion or a stack both work. Do not forget to mark cells before recursing, or you will loop forever.

</details>

In [ ]:
def num_islands(grid):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def num_islands(grid):
    # Approach: scan every cell. Each time an unvisited land cell appears, it starts a new island.
    # Flood-fill the whole island with an explicit stack, marking cells as seen, so no cell
    # is counted twice. An explicit stack avoids Python's recursion limit on large grids.
    # Complexity: O(rows * cols) time and space.
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    seen = set()
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "1" or (r, c) in seen:
                continue                  # water, or land already part of a counted island
            count += 1
            stack = [(r, c)]
            seen.add((r, c))
            while stack:
                row, col = stack.pop()
                for nr, nc in ((row + 1, col), (row - 1, col), (row, col + 1), (row, col - 1)):
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1" and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        stack.append((nr, nc))
    return count
```

</details>

## Exercise 2: Course Schedule

**Difficulty:** Medium · about 25 min

You must take `num_courses` courses numbered `0` to `num_courses - 1`. `prerequisites` holds pairs `[a, b]` meaning course `b` must come before course `a`. Return `True` when every course can be taken, which means the prerequisites contain no cycle.

In a pipeline: this is how Airflow and dbt order a DAG of tasks or models: a topological sort runs each one after its upstream dependencies, and any node left over means a dependency cycle, which both tools reject.

**Example 1**

```text
Input:  num_courses = 2, prerequisites = [[1, 0]]
Output: True
```

**Example 2**

```text
Input:  num_courses = 2, prerequisites = [[1, 0], [0, 1]]
Output: False
```

<details>
<summary>💡 Tip</summary>

Treat each prerequisite pair as an edge in a directed graph. Repeatedly remove courses with no remaining prerequisites, and lower the count of each course that depends on them. If some courses can never be removed, they form a cycle.

</details>

In [ ]:
def can_finish(num_courses, prerequisites):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from collections import deque

def can_finish(num_courses, prerequisites):
    # Approach: topological sort (Kahn's algorithm). Courses with no unmet prerequisites can
    # be taken now. Taking a course removes one unmet requirement from each course that needs it.
    # If every course is eventually taken, there is no cycle. A cycle leaves courses waiting forever.
    # Complexity: O(V + E) time and space.
    indegree = [0] * num_courses                  # unmet prerequisites per course
    after = [[] for _ in range(num_courses)]      # courses unlocked by each course
    for course, prerequisite in prerequisites:
        indegree[course] += 1
        after[prerequisite].append(course)
    ready = deque(c for c in range(num_courses) if indegree[c] == 0)
    done = 0
    while ready:
        course = ready.popleft()
        done += 1
        for nxt in after[course]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:                # all its prerequisites are now taken
                ready.append(nxt)
    return done == num_courses
```

</details>

## Exercise 3: Rotting Oranges

**Difficulty:** Medium · about 20 min

`grid` holds `0` (empty), `1` (fresh orange) and `2` (rotten orange). Each minute, every fresh orange next to a rotten one becomes rotten. Return the minutes until no fresh orange is left, or `-1` if some never rot. Return `0` when there are no fresh oranges at the start.

In a pipeline: the same multi-source, layer-by-layer search tells you how many hops downstream each table sits from a set of failed sources, which is the blast radius of an incident.

**Example 1**

```text
Input:  grid = [[2, 1, 1], [1, 1, 0], [0, 1, 1]]
Output: 4
```

**Example 2**

```text
Input:  grid = [[2, 1, 1], [0, 1, 1], [1, 0, 1]]
Output: -1
```

<details>
<summary>💡 Tip</summary>

Put every rotten orange into the queue at the start. Process the queue one layer at a time: each layer is one minute. Count the fresh oranges, and if any remain after the search, return -1. Return 0 right away when none start fresh.

</details>

In [ ]:
def oranges_rotting(grid):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from collections import deque

def oranges_rotting(grid):
    # Approach: multi-source BFS. Put every rotten orange in the queue at once, so the
    # queue expands all sources in parallel. Each round of the loop is one minute.
    # Count fresh oranges up front; if some remain when the queue empties, they can never rot.
    # Complexity: O(rows * cols) time and space.
    rows, cols = len(grid), len(grid[0])
    queue = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                queue.append((r, c))
            elif grid[r][c] == 1:
                fresh += 1
    minutes = 0
    while queue and fresh:
        minutes += 1
        for _ in range(len(queue)):           # one full layer = one minute
            r, c = queue.popleft()
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2          # mark rotten as we enqueue, so it is counted once
                    fresh -= 1
                    queue.append((nr, nc))
    return minutes if fresh == 0 else -1
```

</details>

## Exercise 4: Word Ladder

**Difficulty:** Hard · about 30 min

Change `begin_word` into `end_word` one letter at a time. Every intermediate word must be in `word_list`. Return the number of words in the shortest such sequence, counting both ends, or `0` if none exists.

**Example 1**

```text
Input:  begin_word = "hit", end_word = "cog", word_list = ["hot", "dot", "dog", "lot", "log", "cog"]
Output: 5
```

**Example 2**

```text
Input:  begin_word = "hit", end_word = "cog", word_list = ["hot", "dot", "dog", "lot", "log"]
Output: 0
```

<details>
<summary>💡 Tip</summary>

This is a shortest-path search. Each word is a node, and two words are connected when they differ by one letter. Run a breadth-first search from `begin_word`, trying every one-letter change. Remove words from the set as you visit them so you never revisit one.

</details>

In [ ]:
def ladder_length(begin_word, end_word, word_list):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from collections import deque

def ladder_length(begin_word, end_word, word_list):
    # Approach: breadth-first search over words. Each step changes one letter, so the first
    # time we reach end_word is the shortest sequence. Trying all 26 letters at each position
    # generates neighbours without comparing every pair of words.
    # Visited words are removed from the set, so each word is queued once.
    # Complexity: O(N * L * 26) time for N words of length L, O(N * L) space.
    words = set(word_list)
    if end_word not in words:
        return 0                              # no path can end at an unlisted word
    queue = deque([(begin_word, 1)])          # the count includes begin_word itself
    while queue:
        word, steps = queue.popleft()
        if word == end_word:
            return steps
        for i in range(len(word)):
            for letter in "abcdefghijklmnopqrstuvwxyz":
                candidate = word[:i] + letter + word[i + 1:]
                if candidate in words:
                    words.remove(candidate)   # visited: never enqueue it again
                    queue.append((candidate, steps + 1))
    return 0
```

</details>

## Exercise 5: Number of connected components

**Difficulty:** Medium · about 20 min

A graph has `n` nodes numbered `0` to `n - 1`, and `edges` lists the connections. Return how many separate connected groups there are.

In a pipeline: union-find is how entity resolution merges records that share an email, phone or device ID into one customer, and how lineage tools group tables into independent subgraphs.

**Example 1**

```text
Input:  n = 5, edges = [[0, 1], [1, 2], [3, 4]]
Output: 2
```

**Example 2**

```text
Input:  n = 5, edges = [[0, 1], [1, 2], [2, 3], [3, 4]]
Output: 1
```

<details>
<summary>💡 Tip</summary>

Start with every node in its own group, so the count is n. Each edge that joins two different groups merges them and lowers the count by one. Path compression in `find` keeps these lookups fast.

</details>

In [ ]:
def count_components(n, edges):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def count_components(n, edges):
    # Approach: union-find. Start with n separate groups. Each edge that joins two different
    # groups merges them and removes one group from the count.
    # find() follows parent links to the group's root, with path halving to keep trees shallow.
    # Complexity: about O(E * alpha(n)) time, O(n) space.
    parent = list(range(n))                   # each node starts as its own root

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]   # path halving: skip a level as we go
            node = parent[node]
        return node

    components = n
    for a, b in edges:
        root_a, root_b = find(a), find(b)
        if root_a != root_b:                  # different groups: merge them
            parent[root_a] = root_b
            components -= 1
    return components
```

</details>